In [15]:
%matplotlib inline
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np
#from neurodynex3.leaky_integrate_and_fire import LIF
#rom neurodynex3.tools import input_factory, plot_tools


# parameters in SI units: volts, seconds, ohms, amps
V_rest  = -0.070    # -70 mV
V_reset = -0.065    # -65 mV
V_th    = -0.050    # -50 mV  firing threshold
R       = 10e6      # 10 MΩ
tau     = 8e-3      # 8 ms  membrane time constant
t_refr  = 2e-3      # 2 ms  absolute refractory period
dt = 1e-4;
rng = np.random.default_rng(0)

tau_syn = 5e-3   # 5 ms
gbar    = 40e-12 # 40 pS


def simulate_lif(I, dt, V_rest,V_reset,V_th, R, tau, t_refr):
    #simulate LIF model through differential equation itself
    n = len(I)
    V=np.zeros(n)
    V[0] = V_rest
    spike_times = []
    refr_steps = int(round(t_refr / dt))   # how many steps to stay clamped
    refr_counter = 0                       # >0 means we're inside refractory

    for k in range(n-1):
        if refr_counter>0:
            V[k+1] = V_reset
            refr_counter -= 1
            continue 
        #differential equation
        dV = (dt / tau) * (-(V[k] - V_rest) + R * I[k])
        V[k+1] = V[k] + dV

        # record spike timings 

        if V[k] < V_th and V[k+1] >= V_th:
            spike_times.append((k+1) * dt)
            V[k+1] = V_reset #reset
            refr_counter = refr_steps
    t = np.arange(n)* dt 
    return t, V, spike_times


def poisson_encode(rate, T, dt, rng):
    n = int(round(T / dt))
    spikes = (np.random.rand(n) < rate * dt).astype(float) #IDENTIFY how many spikes actually occur per timestep
    return spikes

def conductance_trace(spk, gbar, tau_syn, dt):
    #function to determine change in conductance in postsynaptic neuron
    n = len(spk)
    g = np.zeros(n)
    decay = np.exp(-dt / tau_syn)
    for k in range(1, n):
        g[k] = g[k-1] * decay + gbar * spk[k]
    return g 

In [16]:
#testing spike encoding
spk = poisson_encode(100, 0.5, 1e-4, rng)
print(spk.sum(), "spikes in 0.5s at 100Hz — expected ~50")

63.0 spikes in 0.5s at 100Hz — expected ~50


In [17]:
#testing conductance trace

g = conductance_trace(spk, gbar, tau_syn, dt)
print(f"peak conductance: {g.max()*1e12:.1f} pS")

peak conductance: 125.3 pS


In [18]:
# Wiring the conductance-based synapse (Ch.3) into the LIF neuron (soma)
# Pipeline: Poisson spike train -> conductance g(t) -> LIF driven by I_syn = g(t)*(V - E_syn)
# This replaces the fixed current I we used in single-neuron tests with a synaptic current

E_syn   = 0.0     # excitatory reversal potential (~0 mV, Ch.3)
tau_syn = 5e-3    # synaptic decay time constant, 5 ms
gbar    = 1e-7    # conductance bump per spike, scaled up for a single test synapse

def lif_conductance(g, dt, V_rest, V_reset, V_th, R, tau, t_refr, E_syn):
    # same Euler LIF loop as before, but driven by synaptic current instead of fixed I
    n = len(g)
    V = np.zeros(n)
    V[0] = V_rest
    spike_times = []
    refr_steps = int(round(t_refr / dt))
    refr_counter = 0

    for k in range(n - 1):
        if refr_counter > 0:
            V[k+1] = V_reset
            refr_counter -= 1
            continue
        # synaptic current: I_syn = g(t) * (V - E_syn)
        dV = (dt / tau) * (-(V[k] - V_rest) - R * g[k] * (V[k] - E_syn))
        V[k+1] = V[k] + dV
        if V[k] < V_th and V[k+1] >= V_th:
            spike_times.append((k+1) * dt)
            V[k+1] = V_reset
            refr_counter = refr_steps

    t = np.arange(n) * dt
    return t, V, spike_times

# quick test: 100 Hz Poisson input for 0.5s, through the synapse, into the LIF
spk = poisson_encode(100, 0.5, dt, rng)
g   = conductance_trace(spk, gbar, tau_syn, dt)
t, V, spikes = lif_conductance(g, dt, V_rest, V_reset, V_th, R, tau, t_refr, E_syn)

print(f"input: 100 Hz Poisson, {int(spk.sum())} input spikes")
print(f"output: {len(spikes)} output spikes ({len(spikes)/0.5:.1f} Hz)")

input: 100 Hz Poisson, 52 input spikes
output: 41 output spikes (82.0 Hz)


so currently we have poisson_encode which turns target firing rate into a Bernoulli spike train based on the example in 3.1.2
we then have conductance_trace function which turns a spike train into an exponentially-decaying conductance g(t)
the LIF_conductance function is an LIF neuron driven by I_syn = g(t) * (V - E_syn) instead of fixed current, replicating what happens with an actual input from a synapse
In my mind, I think the best approach considering the limited time is just trying to create an untrained feedforward SNN classifier prototype with LIF neurons. If I have time I will apply LIF.

The approach I will use is as follows:

Multi - neuron poisson encoding - essentially taking a feature vector as an input and matching that into spike trains. Each feature maps to a rate (e.g. rate_i = x_i * max_rate), and each rate gets its own independent Poisson spike train
Per output sum conductance - each output neuron o receives input from all N_in input neurons, each connection scaled by a weight W[i, o]. The conductance driving neuron o at time t is the weighted sum over all input neurons' individual conductance traces. This creates synapses with stronger or weaker connections based on the weighting on those synapses. 
Multi-output LIF layer -  each output neuron is its own independent LIF, each with its own V, its own refractory counter, its own spike list — but all driven by the appropriately-weighted-and-summed conductance. 


I will put the above three into one function to create the forward pass of the network, then design the spike count readout to determine which neuron fired the most allowing me to see the decisions the matrix determines

In [20]:
""" 
Here I am designing the poisson encoding function which encodes a feature vector x into N independent Poisson spike trains, 
here for simplicity and time constraints, I will assume the feature values are scaled to [0,1]

"""

def poisson_encode_population(x, max_rate, T, dt, rng):
    x = np.asarray(x)
    N_in = len(x)
    n_steps = int(round(T / dt))

    rates = x * max_rate #linearly mapping each feature to a rate ranging between [0,max_rate]

    spikes = np.zeros((N_in, n_steps))
    for i in range(N_in):
        spikes[i] = (rng.random(n_steps) < rates[i] * dt).astype(float) 
        #i decided to just implement my old function directly instead of calling it, just cleaner.
        

    return spikes, rates

#one of my worries is that I will end up with neurons having the same output 

rng = np.random.default_rng(0)
x = np.array([0.8, 0.8, 0.2])   # two neurons with the SAME rate, one different
spikes, rates = poisson_encode_population(x, max_rate=100, T=0.5, dt=1e-4, rng=rng)

print("rates:", rates)
print("spike counts per neuron:", spikes.sum(axis=1))

# The two neurons with equal rate (rows 0 and 1) should have similar but NOT identical spike counts/timing 
print("rows 0 and 1 identical?", np.array_equal(spikes[0], spikes[1]))

#they dont!!



rates: [80. 80. 20.]
spike counts per neuron: [39. 32. 11.]
rows 0 and 1 identical? False


here, we can see a rate of 80hz should give us spike counts of 40 (t=0.5), and i got 39 and 32, which is within teh standard deviatioon range. I believe this should be fine given it is a poisson process (variance = mean spike count so any spike count within standard deviation should be good)


An LIF neuron leaks current, as it always tries to reset back to a resting voltage. However, its input is a synapse feeding a synaptic current, which is based on the conductance of the synapse and  the difference between the current voltage and some reference value E_syn. This makes the synaptic current being Isyn(t) = g_syn(t)(u(t) - E_syn).
In a SNN, there are multiple input synapses feeding into one neuron. The total input has to consider that the difference (u(t) - E_syn will be a constant term, but the flow of current will be different for each synapse, meaning the synaptic conductance of each synapse is added, then multiply by the difference term. 

I​(t)=j∑​f∑​w_ij​g_ij​(t−tj(f)​)(ui​(t)−Esyn​)

in order to differentiate between synapses, we will add fixed weights to each synapse and see what happens.

Below I will combine N independent input spike trains into one conductance trace per output neuron, using a weight matrix


In [23]:
def combined_conductance(spikes,W,gbar,tau_syn, dt):
    """
    W - weight matrix, scales how strongly inptu neuron i drives output neuron o,
    gbar: baseline conductance jump per spike - gives a number to multiply the weights with
    tau_syn: synaptic decay time constant
    return:
    g_out - one summed conductance trace
    """
    N_in, n_steps = spikes.shape
    N_out = W.shape[1]
    g_out = np.zeros((N_out, n_steps))

    for o in range(N_out):
        #For this output neuron, sum the weighted conductance contribution of every input neuron which is connected
        for i in range(N_in):
            if W[i,o] == 0:
                continue #no synapse so just skip the neuron

            g_out[o] += conductance_trace(spikes[i], gbar * W[i, o], tau_syn, dt)

    return g_out

#now I will test whether it actually accurately adds up two neuron's conductance accurately
# 2 input neurons -> 1 output neuron, different weights
rng = np.random.default_rng(1)
spikes, _ = poisson_encode_population(np.array([0.8, 0.5]), max_rate=100, T=0.5, dt=1e-4, rng=rng)
W = np.array([[1.0], [0.5]])  # neuron 0 full weight, neuron 1 half weight

g_combined = combined_conductance(spikes, W, gbar=1e-7, tau_syn=5e-3, dt=1e-4)

# manually compute the two individual contributions and check they sum to g_combined
g0 = conductance_trace(spikes[0], 1e-7 * 1.0, 5e-3, 1e-4)
g1 = conductance_trace(spikes[1], 1e-7 * 0.5, 5e-3, 1e-4)

print("matches manual sum:", np.allclose(g_combined[0], g0 + g1), g_combined[0], g0, g1)


matches manual sum: True [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 1.65700713e-08
 1.62419619e-08 1.59203495e-08] [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 1.06824291e-08
 1.04709029e-08 1.02635651e-08] [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 5.88764216e-09
 5.77105904e-09 5.65678441e-09]


Since they match, it shows that the conductance is adding up correctly. Now I will begin building the multi-output LIF layer. Each output neuron is its own independent LIF, all driven by the weighted and summed conductance.

In [26]:
def lif_conductance_population(g_out, dt, V_rest, V_reset, V_th, R, tau, t_refr, E_syn):
    # running N independent LIF neurons, each driven by its own combined conductance trace from combined conductance
    """
        Returns
    -------
    V_all      : (N_out, n_steps) array -- voltage trace per neuron
                 (fixed shape, fine as a plain array since every neuron
                 runs for the same number of steps)
    spikes_all : list of length N_out, where spikes_all[o] is the list of
                 spike times for output neuron o
    """

    N_out, n_steps = g_out.shape
    V_all = np.zeros((N_out, n_steps)) 
    spikes_all = [] 

    for o in range(N_out):
        #each lif neuron gets its own independent variables to ensure they are all different
        t, V, spikes = lif_conductance(
            g_out[o], dt, V_rest, V_reset, V_th, R, tau, t_refr, E_syn
        )
        V_all[o] = V
        spikes_all.append(spikes)

    return V_all, spikes_all

#here I will test a singular neuron to see what happens

V_all, spikes_all = lif_conductance_population(
    g_combined, dt, V_rest, V_reset, V_th, R, tau, t_refr, E_syn
)

for o in range(len(spikes_all)):
    print(f"output neuron {o}: {len(spikes_all[o])} spikes")

output neuron 0: 40 spikes


one output neuron, drivved by two weighted inputs computes 40 spikes over 0.5 seconds. I am unsure whether this is a good result or not, but I will assume it is.